# 01 · 数据清洗与加工

**目标**：把 `data/raw/` 下 8 张原始表清洗、加工为可直接分析的分析宽表，输出到 `data/processed/`。

**对应脚本**：`scripts/clean_data.py`（可重复执行，保证口径一致）

**本 notebook 展示 5 步**：
1. 读取原始表并概览
2. 缺失值核查与处理
3. 异常值核查
4. 派生指标（有效订单 / 交付周期 / 客单分层 / 旺季 / 首单标记）
5. 客户 RFM 分群 → 输出分析宽表

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(".").resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent          # 兼容从 notebooks/ 目录启动

RAW = PROJECT / "data/raw"
OUT = PROJECT / "data/processed"
TODAY = pd.Timestamp("2026-08-09")    # 数据截止日

# 读取原始表
products  = pd.read_csv(RAW / "products.csv")
stores    = pd.read_csv(RAW / "stores.csv")
reps      = pd.read_csv(RAW / "sales_reps.csv")
customers = pd.read_csv(RAW / "customers.csv")
orders    = pd.read_csv(RAW / "orders.csv", parse_dates=["order_date", "planned_delivery_date", "actual_delivery_date"])
items     = pd.read_csv(RAW / "order_items.csv")
payments  = pd.read_csv(RAW / "payments.csv", parse_dates=["payment_date"])
after     = pd.read_csv(RAW / "after_sales.csv", parse_dates=["report_date"])

for name, df in [("products", products), ("stores", stores), ("sales_reps", reps),
                 ("customers", customers), ("orders", orders), ("order_items", items),
                 ("payments", payments), ("after_sales", after)]:
    print(f"{name:.<12s} {len(df):>6,} 行 × {df.shape[1]} 列")

products....     48 行 × 15 列
stores......      3 行 × 6 列
sales_reps..      6 行 × 7 列
customers...  1,500 行 × 17 列
orders......  2,401 行 × 24 列
order_items.  3,087 行 × 13 列
payments....  5,982 行 × 7 列
after_sales.    189 行 × 9 列


### 2. 缺失值核查

先看哪些字段有缺失，判断是**数据质量问题**还是**业务属性**。

In [2]:
def miss_report(df, name):
    m = df.isna().mean().sort_values(ascending=False)
    m = m[m > 0]
    if len(m) == 0:
        print(f"{name}: 无缺失值")
    else:
        print(f"{name} 缺失率：\n{m.round(3).to_string()}\n")

miss_report(orders, "orders")
miss_report(customers, "customers")
miss_report(items, "order_items")

orders 缺失率：
is_on_time              0.092
actual_delivery_date    0.092

customers 缺失率：
first_order_date    0.211
last_order_date     0.211
age                 0.181
register_date       0.031

order_items: 无缺失值


**判断**：
- `orders`：`actual_delivery_date` / `actual_lead_days` / `is_on_time` 对**未交付**订单（生产中/待发货/已取消）必然缺失——业务属性，不是脏数据。
- `customers`：企业客户（装修公司/工程客户）没有 `age` / `gender`——业务属性。
- `order_items`：无缺失。

处理原则：**不删行、不随意填值**，用派生标记保留业务含义。

In [3]:
o = orders.copy()
# 1) 未交付订单的实际交付字段：保留 NaN（业务含义），is_on_time 用 -1 显式标记"未交付"
o["is_on_time"] = o["is_on_time"].fillna(-1).astype(int)
# 2) 实际交付周期（天）：仅对"已完成"订单有意义
o["actual_lead_days"] = (o["actual_delivery_date"] - o["order_date"]).dt.days
o.loc[o["order_status"] != "已完成", "actual_lead_days"] = np.nan
print("已处理：is_on_time(-1=未交付) / actual_lead_days(仅已完成有值)")

已处理：is_on_time(-1=未交付) / actual_lead_days(仅已完成有值)


### 3. 异常值核查

In [4]:
print("负数金额检查：")
print("  final_amount<0:", (o["final_amount"] < 0).sum(), " 行")
print("  line_amount <0:", (items["line_amount"] < 0).sum(), " 行")

# 单价合理性：单位面积价格（元/㎡）分布，识别离群
it = items.merge(products[["product_id", "base_price", "defect_rate"]], on="product_id", how="left")
it["unit_price_pm"] = it["unit_price"] / it["area_m2"]
print("\n单位面积价格（元/㎡）分布：")
print(it["unit_price_pm"].describe().round(0).to_string())
print("\n单价 >5万元/樘 的极端样本数：", (it["unit_price"] > 50000).sum())

负数金额检查：
  final_amount<0: 0  行
  line_amount <0: 0  行

单位面积价格（元/㎡）分布：
count    3087.0
mean     4135.0
std      2081.0
min       626.0
25%      2403.0
50%      3795.0
75%      6148.0
max      8010.0

单价 >5万元/樘 的极端样本数： 0


### 4. 派生指标（口径统一）

In [5]:
o["is_valid"] = (o["order_status"] != "已取消").astype(int)     # 有效订单
o["discount_amount"] = o["total_amount"] * o["discount"]           # 让利金额
o["year_month"] = o["order_date"].dt.to_period("M")
o["is_peak_season"] = o["order_month"].isin([3, 4, 5, 9, 10, 11]).astype(int)  # 装修旺季
o["order_tier"] = pd.cut(o["final_amount"], bins=[0, 1e4, 3e4, 6e4, np.inf],
                         labels=["<1万", "1-3万", "3-6万", ">6万"], right=False).astype("string")

# 首单标记（每个客户第一次下单，用于"新客/回头客"）
fd = o.sort_values("order_date").drop_duplicates("customer_id", keep="first")[["customer_id", "order_date"]]
fd = fd.rename(columns={"order_date": "cust_first_order"})
o = o.merge(fd, on="customer_id", how="left")
o["is_first_order"] = (o["order_date"] == o["cust_first_order"]).astype(int)

# 关联门店 / 业务员名称
o = o.merge(stores[["store_id", "store_name", "store_type"]], on="store_id", how="left")
o = o.merge(reps[["rep_id", "rep_name"]], left_on="sales_rep_id", right_on="rep_id", how="left")
print("orders_clean 维度：", o.shape)

orders_clean 维度： (2401, 36)


### 5. 客户 RFM 分群

In [6]:
valid = o[o["is_valid"] == 1]
rfm = valid.groupby("customer_id").agg(
    last_order=("order_date", "max"),
    frequency=("order_id", "nunique"),
    monetary=("final_amount", "sum")).reset_index()
rfm["recency_days"] = (TODAY - rfm["last_order"]).dt.days

# 四分位打分（1~4，越大越好）；recency 天数越少越好
def score_q(s, reverse=False):
    qs = s.quantile([.25, .5, .75])
    if reverse:
        return s.apply(lambda v: 1 if v >= qs[.75] else (2 if v >= qs[.5] else (3 if v >= qs[.25] else 4)))
    return s.apply(lambda v: 1 if v <= qs[.25] else (2 if v <= qs[.5] else (3 if v <= qs[.75] else 4)))

rfm["R"] = score_q(rfm["recency_days"], reverse=True)
rfm["F"] = score_q(rfm["frequency"])
rfm["M"] = score_q(rfm["monetary"])
rfm["rfm_score"] = rfm["R"] + rfm["F"] + rfm["M"]

def seg(r):
    R, F, M = r["R"], r["F"], r["M"]
    if F >= 3 and M >= 3 and R >= 3: return "高价值客户"
    if R >= 3 and M >= 3:            return "成长潜力"
    if F >= 2 and R < 3:             return "需挽留"
    if R >= 3:                       return "新客户"
    return "沉睡客户"

rfm["segment"] = rfm.apply(seg, axis=1)
rfm = rfm.merge(customers[["customer_id", "customer_name", "customer_type",
                           "channel", "city", "member_level"]], on="customer_id", how="left")
rfm["segment"].value_counts().reindex(["高价值客户", "成长潜力", "需挽留", "新客户", "沉睡客户"])

segment
高价值客户    213
成长潜力     161
需挽留      203
新客户      215
沉睡客户     391
Name: count, dtype: int64

### 6. 输出清洗后的分析宽表

In [7]:
it = it.merge(o[["order_id", "order_date", "order_status", "order_type", "customer_type"]],
                   on="order_id", how="left")

o.sort_values("order_date").to_csv(OUT / "orders_clean.csv", index=False, encoding="utf-8-sig")
it.to_csv(OUT / "item_flat.csv", index=False, encoding="utf-8-sig")
rfm.to_csv(OUT / "customer_rfm.csv", index=False, encoding="utf-8-sig")
print("已输出到 data/processed/：")
print("  orders_clean.csv / item_flat.csv / customer_rfm.csv")

已输出到 data/processed/：
  orders_clean.csv / item_flat.csv / customer_rfm.csv


### 清洗小结
- **有效订单**：2,401 单剔除 **169 单取消** → **2,232 单**（`is_valid=1`）
- **缺失值**：交付类字段缺失是"未交付"业务属性，保留；企业客户无年龄/性别，保留
- **异常值**：金额无负数；单位面积价格分布合理，无极端离群
- **派生指标**：交付周期、客单分层、旺季、首单标记、RFM 分群已就绪

> 口径约定见 `docs/data_dictionary.md`；清洗逻辑已沉淀为 `scripts/clean_data.py`，可一键重跑。